# Contig filtering and FASTA export

Takes the Snakemake pipeline's combined MEGAN6 output (assembled contigs
across *every* sequenced SRR, including control/debug samples) and narrows
it down to the curated candidate set used for manual curation.

Steps:
0. Import packages and data (Snakemake final output table)
1. Remove control SRRs (keep only the 253 selected SRRs from `dataset_discovery/samples.tsv`)
2. Detect duplicated contigs (same underlying dataset submitted under multiple SRA accessions), keep the copy with better metadata
3. Remove contigs classified as *cellular organisms* by MEGAN taxonomy
4. Remove the SRR6846476 dataset (manually flagged as anomalous: chimeric sequences, unusually short contigs)
5. Check numbers
6. Write FASTA files

Ends at **510 curated candidate contigs from 131 SRRs** — the set the paper
reports manual curation was performed on.

In [5]:
import pandas as pd
from pysradb.sraweb import SRAweb
from Bio import SeqIO

## 0. Import data

In [6]:
# Snakemake pipeline output: combined MEGAN6 results across *every* sequenced SRR,
# including control/debug samples that were never part of the 253-SRR selection.
# Assumes a sibling checkout of tobamo-snakemake that has been run over
# dataset_discovery/samples.tsv -- adjust the path if your checkout lives elsewhere.
df = pd.read_csv('../../data/snakemake/megan6_results_combined.csv', index_col=0)

# the 253 selected SRRs
samples_path = '../../dataset_discovery/samples.tsv'
with open(samples_path) as file:
    samples = [line.strip() for line in file.readlines()][1:]

## 1. Remove control SRRs

> This step is bookkeeping, not a decision you need to make. The Snakemake
> output above still contains the control/debug SRRs used to validate the
> pipeline itself — they were never part of the 253-SRR selection and have
> already been removed from this repo, since they're not used anywhere
> downstream. This cell just aligns the Snakemake table with `samples.tsv`;
> every SRR of interest is already present in `samples`.

In [7]:
# keep only the 253 selected samples, drop everything else (control/debug SRRs)
test_results = df[df['SRR'].isin(samples)]

## 2. Remove duplicated contigs

Same underlying dataset can be submitted to SRA under multiple accessions.
Detect contigs whose sequence is shared across more than one `qseqid`/SRR,
then decide (by manual inspection of metadata) which SRA entry to keep.

In [8]:
# Group by sequence and collect unique qseqid values for each sequence
seq_to_qseqid = test_results.groupby('sequence')['qseqid'].unique()

# Find sequences that are associated with more than one unique qseqid
duplicated_seqs = seq_to_qseqid[seq_to_qseqid.apply(len) > 1]

# Get all qseqid values involved in duplicated sequences
duplicated_qseqids = set(qseqid for qseqids in duplicated_seqs for qseqid in qseqids)

print(f"Number of sequences shared by multiple qseqid: {len(duplicated_seqs)}")
print(f"Number of qseqid involved: {len(duplicated_qseqids)}")
print("qseqid involved in duplicated sequences:")
print(duplicated_qseqids)

srr_list = test_results[test_results['qseqid'].isin(duplicated_qseqids)]['SRR'].unique().tolist()
print('SRRs with duplicated qseqids:')
print(srr_list)

Number of sequences shared by multiple qseqid: 15
Number of qseqid involved: 31
qseqid involved in duplicated sequences:
{'NODE_138_length_1097_cov_1637.345361_ERR2737479', 'NODE_263_length_789_cov_170.131420_ERR3179625', 'NODE_14_length_5191_cov_521.552923_ERR3179625', 'NODE_83_length_1545_cov_832.384344_ERR3179625', 'NODE_186_length_6730_cov_324.596573_SRR8749694', 'NODE_121_length_1179_cov_560.262357_ERR2737479', 'NODE_263_length_789_cov_170.131420_ERR2737479', 'NODE_4269_length_2337_cov_77.980719_SRR6233765', 'NODE_80_length_1581_cov_92.495186_ERR2737479', 'NODE_228_length_845_cov_1276.600279_ERR2737479', 'NODE_244_length_5617_cov_62.071480_SRR8658357', 'NODE_14_length_5191_cov_521.552923_ERR2737479', 'NODE_186_length_6730_cov_324.596573_SRR8658358', 'NODE_83_length_1545_cov_832.384344_ERR2737479', 'NODE_316_length_714_cov_3.042589_ERR3179625', 'NODE_4542_length_2337_cov_155.333041_SRR5087405', 'NODE_82_length_1552_cov_1.965614_ERR2737479', 'NODE_138_length_1097_cov_1637.345361_ERR

> **Manual inspection step:** look up the involved SRRs online, or via their
> SRA metadata below, to decide which duplicate entry to keep.

In [9]:
# RUN ONLY ONCE -- hits the SRA API
# db = SRAweb()
# metadata = db.sra_metadata(srr_list, detailed=True)
# metadata.to_csv('../results/metadata_of_duplicated_contigs.csv')

metadata = pd.read_csv('../results/metadata_of_duplicated_contigs.csv', index_col=0)
metadata

,run_accession,study_accession,study_title,experiment_accession,experiment_title,experiment_desc,organism_taxid,organism_name,library_name,library_strategy,...,cell_type,sample_type,treatment,replicate,ena_fastq_http,ena_fastq_http_1,ena_fastq_http_2,ena_fastq_ftp,ena_fastq_ftp_1,ena_fastq_ftp_2
0,ERR2737479,ERP108694,Virus Discovery for Vietnam Initiative on Zoon...,ERX2750552,Illumina HiSeq 2500 paired end sequencing; Ill...,Illumina HiSeq 2500 paired end sequencing; Ill...,1070528,viral metagenome,NaN,WGS,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/ERR273/009...,http://ftp.sra.ebi.ac.uk/vol1/fastq/ERR273/009...,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/ERR273/...,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/ERR273/...
1,ERR3179625,ERP006046,Virus_Discovery_for_Vietnam_Initiative_on_Zoon...,ERX3207476,Illumina HiSeq 2500 paired end sequencing,Illumina HiSeq 2500 paired end sequencing,1070528,viral metagenome,DN459406Q:F12,WGS,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/ERR317/005...,http://ftp.sra.ebi.ac.uk/vol1/fastq/ERR317/005...,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/ERR317/...,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/ERR317/...
2,SRR5087400,SRP094756,Gene expression changes after microenvironment...,SRX2404651,"P493-6 cell line, untreated","P493-6 cell line, untreated",9606,Homo sapiens,P493-6 untreated replicate 5,RNA-Seq,...,B-cell,cells,none,Biological replicate 5,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR508/000...,NaN,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR508/...,NaN,NaN
3,SRR5087405,SRP094756,Gene expression changes after microenvironment...,SRX2404656,"P493-6 cell line, dox-treated, environmental s...","P493-6 cell line, dox-treated, environmental s...",9606,Homo sapiens,P493-6 combination 3,RNA-Seq,...,B-cell,cells,"1 ng/ml doxycycline for 16h; CpG (0.5muM, ODN2...",Biological replicate 1,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR508/005...,NaN,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR508/...,NaN,NaN
4,SRR6233765,SRP094756,Gene expression changes after microenvironment...,SRX3342207,"P493-6 cell line, a-IgM treated","P493-6 cell line, a-IgM treated",9606,Homo sapiens,P493-6 a-IgM replicate 1,RNA-Seq,...,B-cell,cells,"a-IgM F(ab)2 fragments (130ng/ml, Jackson Immu...",Biological replicate 1,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR623/005...,NaN,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR623/...,NaN,NaN
5,SRR8658357,SRP187337,Panonychus citri Genome sequencing,SRX5456065,RNA-Seq of Panonychus citri: susceptible_3,RNA-Seq of Panonychus citri: susceptible_3,50023,Panonychus citri,Pc_SUS_3,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/007...,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/007...,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...
6,SRR8658358,SRP187337,Panonychus citri Genome sequencing,SRX5456064,RNA-Seq of Panonychus citri: susceptible_2,RNA-Seq of Panonychus citri: susceptible_2,50023,Panonychus citri,Pc_SUS_2,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/008...,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/008...,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...
7,SRR8658359,SRP187337,Panonychus citri Genome sequencing,SRX5456063,RNA-Seq of Panonychus citri: susceptible_1,RNA-Seq of Panonychus citri: susceptible_1,50023,Panonychus citri,Pc_SUS_1,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/009...,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR865/009...,NaN,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...,era-fasp@fasp.sra.ebi.ac.uk:vol1/fastq/SRR865/...
8,SRR8749693,SRP188804,Panonychus citri Raw sequence reads,SRX5540672,RNA-Seq of Panonychus citri: susceptible_no-in...,RNA-Seq of Panonychus citri: susceptible_no-in...,50023,Panonychus citri,Pc_SUS_non1,RNA-Seq,...,NaN,NaN,NaN,NaN,NaN,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR874/003...,http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR874/003...

In [10]:
# after manual metadata inspection, remove duplicated SRRs (same study, different SRA entries), keep only one copy (with better metadata)
SRRs_to_remove = ['ERR2737479', 'SRR8749694', 'SRR8749695', 'SRR8749693', 'SRR5087405', 'SRR6233765']

test_results_deduplicated = test_results[~test_results['SRR'].isin(SRRs_to_remove)]

## 3. Remove cellular organisms

As classified by MEGAN taxonomy.

In [11]:
test_results_non_cellular = test_results_deduplicated[~test_results_deduplicated['megan_tax'].str.contains('cellular')]
test_results_cellular = test_results_deduplicated[test_results_deduplicated['megan_tax'].str.contains('cellular')]

## 4. Remove problematic SRR6846476

Manually flagged as anomalous during curation (chimeric sequences,
unusually short contigs).

In [12]:
test_results_non_cellular_filtered = test_results_non_cellular[~(test_results_non_cellular['SRR'] == 'SRR6846476')]

## 5. Check numbers

In [13]:
print("Contig counts report:")
print(f"1. All contigs: {df.qseqid.nunique()}")
print(f"2. After removing control SRRs: {test_results.qseqid.nunique()}")
print(f"3. After removing duplicated SRRs: {test_results_deduplicated.qseqid.nunique()}")
print(f"4A. Non-cellular contigs: {test_results_non_cellular.qseqid.nunique()}")
print(f"4B. Cellular contigs: {test_results_cellular.qseqid.nunique()}")
print(
    f"5. Final curated set: {test_results_non_cellular_filtered.qseqid.nunique()} contigs "
    f"from {test_results_non_cellular_filtered.SRR.nunique()} SRRs"
)

Contig counts report:
1. All contigs: 3406
2. After removing control SRRs: 2567
3. After removing duplicated SRRs: 2549
4A. Non-cellular contigs: 2383
4B. Cellular contigs: 166
5. Final curated set: 510 contigs from 131 SRRs


## 6. Write FASTA files

In [14]:
# make a dict from qseqid and sequence
contigs_all = dict(df.filter(['qseqid', 'sequence']).values) # all contigs
contigs_all_test = dict(test_results.filter(['qseqid', 'sequence']).values) # test contigs (removed control SRRs)
contigs_all_deduplicated = dict(test_results_deduplicated.filter(['qseqid', 'sequence']).values) # deduplicated contigs
contigs_non_cellular = dict(test_results_non_cellular.filter(['qseqid', 'sequence']).values) # non-cellular contigs
contigs_non_cellular_filtered = dict(test_results_non_cellular_filtered.filter(['qseqid', 'sequence']).values) # final curated set

In [15]:
def write_fasta(seq_dict, output_file):
    with open(output_file, 'w') as o:
        for key, val in seq_dict.items():
            o.write('>' + key + '\n')
            o.write(val + '\n')
    return output_file

In [16]:
# make fasta files, skip if file already exists
path = '../../data/contigs/'

for seq_dict, output_file in [
    (contigs_all, f'{path}contigs_all.fasta'),
    (contigs_all_test, f'{path}contigs_all_test.fasta'),
    (contigs_all_deduplicated, f'{path}contigs_all_deduplicated.fasta'),
    (contigs_non_cellular, f'{path}contigs_non_cellular.fasta'),
    (contigs_non_cellular_filtered, f'{path}contigs_non_cellular_filtered.fasta'),
]:
    write_fasta(seq_dict, output_file)
    print(f"Wrote {output_file}")

Wrote ../../data/contigs/contigs_all.fasta
Wrote ../../data/contigs/contigs_all_test.fasta
Wrote ../../data/contigs/contigs_all_deduplicated.fasta
Wrote ../../data/contigs/contigs_non_cellular.fasta
Wrote ../../data/contigs/contigs_non_cellular_filtered.fasta
